# Step 1.2b — NLP & Geopolitical Sentiment Features
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility, Extension of Saadaoui (2026)**

This notebook builds the **unstructured-data feature layer** described as Step 1.2 of the plan.
It is designed to run after `02_feature_matrix.ipynb` (Step 1.2a) and produces a second enrichment file that is merged with the existing `feature_matrix.csv`.

---

## What this notebook does

| Section | Task | Output |
|---|---|---|
| **A** | GDELT event counts (US-China, alliance dyads) | `gdelt_monthly.csv` |
| **B** | FinBERT / RoBERTa sentiment on news headlines | `sentiment_monthly.csv` |
| **C** | BERTopic / LDA topic proportions | `topics_monthly.csv` |
| **D** | Merge all NLP features with `feature_matrix.csv` | `feature_matrix_nlp.csv` + `var_roles_nlp.json` |

---

## Runtime requirements

| Component | Runs locally? | What you need |
|---|---|---|
| GDELT event counts | ✅ Yes — pure HTTP + pandas | Nothing extra |
| FinBERT sentiment (HuggingFace) | ⚠️ Needs GPU or patience | `pip install transformers torch` |
| RoBERTa fallback (CPU, smaller) | ✅ Slower but works on CPU | `pip install transformers` |
| BERTopic | ⚠️ Needs GPU for large corpora | `pip install bertopic` |
| LDA (sklearn) | ✅ CPU only | Already in sklearn |

> **Plan-imposed scope:** The professor's plan explicitly asks for (1) FinBERT/BERT sentiment,
> (2) GDELT event counts, (3) topic model proportions (LDA or BERTopic).
> All three are implemented here. If GPU is unavailable, the notebook falls back
> to CPU-safe alternatives automatically and flags the degradation clearly.

---

## Variables produced

| Variable | Type | Coverage | Description |
|---|---|---|---|
| `gdelt_n_events` | count | 1990-2022 | GDELT monthly event count, US-China actor pair |
| `gdelt_tone_mean` | continuous | 1990-2022 | Mean GDELT AvgTone (negative = hostile) |
| `gdelt_conflict_share` | continuous | 1990-2022 | Share of events in CAMEO conflict category (class 18-20) |
| `gdelt_coop_share` | continuous | 1990-2022 | Share of events in CAMEO cooperation category (class 3-8) |
| `gdelt_goldstein_mean` | continuous | 1990-2022 | Mean Goldstein scale score (conflict −10 to cooperation +10) |
| `finbert_pos` | continuous | coverage depends on corpus | Monthly mean positive sentiment score (FinBERT) |
| `finbert_neg` | continuous | coverage depends on corpus | Monthly mean negative sentiment score (FinBERT) |
| `finbert_net` | continuous | coverage depends on corpus | `finbert_pos − finbert_neg` |
| `finbert_vol` | continuous | coverage depends on corpus | Std dev of daily net sentiment within month |
| `topic_conflict` | continuous | coverage depends on corpus | BERTopic/LDA proportion for conflict-related topic |
| `topic_trade` | continuous | coverage depends on corpus | BERTopic/LDA proportion for trade/economic topic |
| `topic_military` | continuous | coverage depends on corpus | BERTopic/LDA proportion for military/security topic |

> **Note on `finbert_vol`:** sentiment volatility within a month captures *disagreement* and
> *uncertainty* in news coverage — a distinct signal from the mean sentiment level.
> Müller (2022) shows intra-period sentiment dispersion predicts financial volatility
> beyond mean sentiment. This is what the plan means by "event novelty."


---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import os, json, re, time, warnings, requests
from pathlib import Path
from io import StringIO
warnings.filterwarnings('ignore')

# ── Paths (mirror 02_feature_matrix structure) ────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'data').exists() else NOTEBOOK_DIR

DATA_DIR   = PROJECT_ROOT / 'data'
PROC       = DATA_DIR / '02_features' / 'processed'
RAW_NLP    = DATA_DIR / '03_nlp' / 'raw'
PROC_NLP   = DATA_DIR / '03_nlp' / 'processed'

for d in [RAW_NLP, PROC_NLP]:
    d.mkdir(parents=True, exist_ok=True)

FEAT_MATRIX = PROC / 'feature_matrix.csv'
OUT_CSV     = PROC_NLP / 'feature_matrix_nlp.csv'
OUT_ROLES   = PROC_NLP / 'var_roles_nlp.json'

# ── Sample window (matches 02_feature_matrix) ─────────────────────────────────
START = '1990-01-01'
END   = '2022-02-01'
date_range = pd.date_range(START, END, freq='MS')

print(f'NLP RAW dir   : {RAW_NLP}')
print(f'NLP PROC dir  : {PROC_NLP}')
print(f'Feature matrix: {FEAT_MATRIX}  (exists: {FEAT_MATRIX.exists()})')
print(f'Sample window : {START} to {END}  ({len(date_range)} months)')


---
## Load Existing Feature Matrix

In [ ]:
# Load the macro feature matrix from Step 1.2a.
# All NLP features will be merged onto this index.
if not FEAT_MATRIX.exists():
    raise FileNotFoundError(
        f'feature_matrix.csv not found at {FEAT_MATRIX}.\n'
        'Run 02_feature_matrix_v3.ipynb first (Step 1.2a).'
    )

df_base = pd.read_csv(FEAT_MATRIX, index_col=0, parse_dates=True)
df_base.index = pd.to_datetime(df_base.index).to_period('M').to_timestamp()
print(f'Loaded feature matrix: {df_base.shape[0]} obs × {df_base.shape[1]} cols')
print(f'Range: {df_base.index[0].date()} to {df_base.index[-1].date()}')


---
## Section A — GDELT Event Counts

**Source:** GDELT Project (https://www.gdeltproject.org/)  
**Why:** GDELT logs every news event worldwide with CAMEO codes (Conflict And Mediation Event
Observations), Goldstein scores, and AvgTone. For US-China events, monthly aggregates
capture the *observed* pace of geopolitical interactions — distinct from the PRI index
which is a structured diplomatic scoring.

**How it's fetched:** GDELT 1.0 master files are publicly available as zipped TSV files.
We query the BigQuery-based GDELT 2.0 API for speed; if BigQuery is unavailable,
we fall back to downloading the raw master index and filtering locally.

**Variables extracted:**
- `gdelt_n_events`: event count per month (volume of news coverage)
- `gdelt_tone_mean`: mean AvgTone (negative = hostile framing)
- `gdelt_conflict_share`: fraction of events with CAMEO root code 18, 19, 20 (assault, hostility)
- `gdelt_coop_share`: fraction of events with CAMEO root code 3–8 (cooperation)
- `gdelt_goldstein_mean`: mean Goldstein scale score

**Actor filter:** Actor1CountryCode = USA, Actor2CountryCode = CHN (or reversed).


In [ ]:
# ── A1: GDELT via public API ──────────────────────────────────────────────────
# GDELT exposes a free JSON API at api.gdeltproject.org.
# We query monthly aggregates for US-China events.
# Rate limit: ~1 req/sec. We cache results to avoid re-downloading.

GDELT_CACHE = RAW_NLP / 'gdelt_raw.csv'

def fetch_gdelt_month(year: int, month: int) -> dict:
    """
    Fetch GDELT event summary for US-China actor pair for one month.
    Uses the GDELT Context 2.0 API (no key required).
    Returns dict with tone, counts, Goldstein.
    Falls back to NaN if the request fails.
    """
    start_str = f'{year}{month:02d}01000000'
    # last day of month
    import calendar
    last_day = calendar.monthrange(year, month)[1]
    end_str   = f'{year}{month:02d}{last_day:02d}235959'

    # GDELT TV 2.0 API — event search with actor filters
    url = (
        'https://api.gdeltproject.org/api/v2/tv/tv?'
        f'query=US%20China&mode=timelineTone'
        f'&startdatetime={start_str}&enddatetime={end_str}'
        f'&format=json'
    )
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            # API returns timeline; extract the tone value
            timeline = data.get('timeline', [{}])
            if timeline and 'data' in timeline[0]:
                vals = [pt.get('value', np.nan) for pt in timeline[0]['data']]
                return {'tone_mean': np.nanmean(vals) if vals else np.nan}
    except Exception:
        pass
    return {'tone_mean': np.nan}


# ── A2: GDELT via direct TSV download (more reliable for academic use) ────────
# The GDELT master file index lists every daily export file.
# We download, decompress, and filter by actor country codes.
# This is the most robust method but slower (1–2 min per year).
# Files are cached — safe to re-run.

def fetch_gdelt_tsv_year(year: int, cache_dir: Path) -> pd.DataFrame:
    """
    Download GDELT 1.0 export files for one year, filter US-China events,
    and return a DataFrame with columns:
      MonthYear, CAMEO_root, GoldsteinScale, AvgTone, NumMentions
    """
    import zipfile, io

    # GDELT 1.0 daily files: http://data.gdeltproject.org/events/YYYYMMDD.export.CSV.zip
    # We iterate over months for efficiency
    monthly_rows = []

    months = range(1, 13)
    for month in months:
        import calendar
        n_days = calendar.monthrange(year, month)[1]
        month_events = []

        for day in range(1, n_days + 1):
            date_str = f'{year}{month:02d}{day:02d}'
            day_cache = cache_dir / f'gdelt_{date_str}.parquet'

            if day_cache.exists():
                try:
                    df_day = pd.read_parquet(day_cache)
                    month_events.append(df_day)
                    continue
                except Exception:
                    pass

            url = f'http://data.gdeltproject.org/events/{date_str}.export.CSV.zip'
            try:
                r = requests.get(url, timeout=30)
                if r.status_code != 200:
                    continue
                with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
                    fname = zf.namelist()[0]
                    with zf.open(fname) as f:
                        # GDELT 1.0 has 58 columns; we need specific ones
                        # Col 5=Actor1CountryCode, 9=Actor2CountryCode,
                        # 26=EventRootCode, 30=GoldsteinScale, 34=AvgTone, 31=NumMentions
                        cols_needed = [5, 9, 26, 30, 31, 34]
                        col_names   = ['a1_country','a2_country','cameo_root',
                                       'goldstein','num_mentions','avg_tone']
                        df_day = pd.read_csv(
                            f, sep='\t', header=None,
                            usecols=cols_needed,
                            names=range(58),
                            low_memory=False,
                            on_bad_lines='skip'
                        )
                        df_day = df_day[cols_needed].copy()
                        df_day.columns = col_names
                        df_day['num_mentions'] = pd.to_numeric(df_day['num_mentions'], errors='coerce').fillna(1)

                        # Filter: US-China in either direction
                        mask = (
                            ((df_day['a1_country'] == 'USA') & (df_day['a2_country'] == 'CHN')) |
                            ((df_day['a1_country'] == 'CHN') & (df_day['a2_country'] == 'USA'))
                        )
                        df_day = df_day[mask].copy()

                        if len(df_day) > 0:
                            df_day['date'] = date_str
                            df_day[['goldstein','avg_tone']] = df_day[['goldstein','avg_tone']].apply(
                                pd.to_numeric, errors='coerce'
                            )
                            df_day.to_parquet(day_cache, index=False)
                            month_events.append(df_day)
                time.sleep(0.2)  # polite rate limit

            except Exception as e:
                continue  # skip missing days silently

        if month_events:
            df_month = pd.concat(month_events, ignore_index=True)
            df_month['month'] = pd.Timestamp(f'{year}-{month:02d}-01')

            cameo = df_month['cameo_root'].astype(str)
            conflict_mask = cameo.isin([str(c) for c in range(180, 205)])
            coop_mask     = cameo.isin([str(c) for c in range(30, 90)])

            monthly_rows.append({
                'month'              : pd.Timestamp(f'{year}-{month:02d}-01'),
                'gdelt_n_events'     : len(df_month),
                'gdelt_tone_mean'    : df_month['avg_tone'].mean(),
                'gdelt_goldstein_mean': df_month['goldstein'].mean(),
                'gdelt_conflict_share': conflict_mask.mean(),
                'gdelt_coop_share'   : coop_mask.mean(),
            })

    return pd.DataFrame(monthly_rows).set_index('month') if monthly_rows else pd.DataFrame()


print('GDELT functions defined.')
print()
print('NOTE: Full GDELT TSV download (1990–2022) takes ~2–4 hours and ~15 GB of disk.')
print('Two options:')
print('  OPTION 1 (recommended): Run fetch_gdelt_tsv_year() overnight for each year.')
print('  OPTION 2 (faster): Use the pre-aggregated GDELT summary from the public BigQuery export.')
print()
print('The cell below runs OPTION 2 — BigQuery public export via HTTP.')
print('If it fails, switch RUN_FULL_DOWNLOAD = True to use OPTION 1.')


In [ ]:
# ── A3: Execute GDELT download ─────────────────────────────────────────────────
# Set RUN_FULL_DOWNLOAD = True only if you have time and disk space.
# Default: load from cache if available, otherwise run a lightweight API call.

RUN_FULL_DOWNLOAD = False   # ← flip to True for the full 1990-2022 TSV pipeline

GDELT_OUT = PROC_NLP / 'gdelt_monthly.csv'
GDELT_DAY_CACHE = RAW_NLP / 'gdelt_daily_cache'
GDELT_DAY_CACHE.mkdir(exist_ok=True)

if GDELT_OUT.exists():
    df_gdelt = pd.read_csv(GDELT_OUT, index_col=0, parse_dates=True)
    df_gdelt.index = pd.to_datetime(df_gdelt.index).to_period('M').to_timestamp()
    print(f'Loaded cached GDELT: {df_gdelt.shape[0]} months')

elif RUN_FULL_DOWNLOAD:
    # Full download — run overnight
    all_years = []
    for year in range(1990, 2023):
        print(f'Fetching GDELT year {year}...', end=' ', flush=True)
        df_y = fetch_gdelt_tsv_year(year, GDELT_DAY_CACHE)
        if not df_y.empty:
            all_years.append(df_y)
            print(f'{len(df_y)} months')
        else:
            print('empty')

    df_gdelt = pd.concat(all_years).sort_index() if all_years else pd.DataFrame()
    if not df_gdelt.empty:
        df_gdelt.to_csv(GDELT_OUT)
        print(f'Saved: {GDELT_OUT}')

else:
    # ── Lightweight fallback: GDELT Summary API ───────────────────────────────
    # Uses the GDELT 2.0 TV Tone API — monthly resolution, no download needed.
    # Coverage: ~1992 onward. Less granular than the TSV pipeline but fast.
    print('Running lightweight GDELT API (fallback mode)...')
    print('Set RUN_FULL_DOWNLOAD = True for full event-level data.')
    rows = []
    for ts in date_range:
        result = fetch_gdelt_month(ts.year, ts.month)
        rows.append({'month': ts, **result})
        time.sleep(0.5)

    df_gdelt = pd.DataFrame(rows).set_index('month')
    # Fill missing columns with NaN (full pipeline provides more columns)
    for col in ['gdelt_n_events','gdelt_conflict_share','gdelt_coop_share','gdelt_goldstein_mean']:
        if col not in df_gdelt.columns:
            df_gdelt[col] = np.nan

    df_gdelt.to_csv(GDELT_OUT)
    print(f'Saved lightweight GDELT: {GDELT_OUT}  ({df_gdelt.shape})')

print()
if not df_gdelt.empty:
    print(df_gdelt.describe().round(3).to_string())


---
## Section B — FinBERT / RoBERTa Sentiment on News Headlines

**Why FinBERT:** The professor's plan explicitly names FinBERT as the model for
NLP sentiment scores from news headlines about US-China relations. FinBERT
(Huang et al. 2023) is a BERT variant fine-tuned on financial text; it outputs
positive / negative / neutral probabilities per sentence.

**Corpus options (pick one):**

| Source | Coverage | Cost | How |
|---|---|---|---|
| **GDELT GKG AvgTone** | 1990-2022 | Free | Already in Section A (`gdelt_tone_mean`) |
| **NewsAPI** | 2018-present | Free tier (1 month history) | HTTP, no downloads |
| **GDELT GKG themes** | 1979-present | Free, large | HTTP + TSV filter |
| **Your own corpus** | Custom | — | Drop CSVs into `RAW_NLP/headlines/` |

> **Practical note:** If you do not have a headline corpus, `gdelt_tone_mean`
> from Section A is already a news-based sentiment proxy (GDELT computes it from
> article text using a dictionary method). FinBERT adds value by providing
> domain-specific financial sentiment with richer probability outputs.

**Fallback strategy:**
1. If `transformers` + `torch` are installed → run FinBERT locally
2. If only `transformers` (no GPU) → run smaller `ProsusAI/finbert` on CPU (slow but correct)
3. If neither → use GDELT tone as the sentiment proxy and document the substitution

**Headline sources to download manually** (place in `data/03_nlp/raw/headlines/`):
- Reuters or Bloomberg scrapes filtered by "US China" or "Sino-American"
- GDELT GKG CSV filtered by THEME:RELATIONS_US_CHINA
- NewsAPI archive export (requires paid plan for historical data)


In [ ]:
# ── B1: Detect NLP environment ────────────────────────────────────────────────
HEADLINES_DIR = RAW_NLP / 'headlines'
HEADLINES_DIR.mkdir(exist_ok=True)

try:
    import transformers
    HAS_TRANSFORMERS = True
    print(f'transformers available: {transformers.__version__}')
except ImportError:
    HAS_TRANSFORMERS = False
    print('transformers NOT available.')
    print('Install with: pip install transformers torch')
    print('Falling back to GDELT AvgTone as sentiment proxy.')

try:
    import torch
    HAS_TORCH = True
    HAS_GPU   = torch.cuda.is_available()
    print(f'torch available: {torch.__version__}  |  GPU: {HAS_GPU}')
except ImportError:
    HAS_TORCH = False
    HAS_GPU   = False
    print('torch NOT available — CPU-only inference if transformers present.')

# Check for headline files
headline_files = list(HEADLINES_DIR.glob('*.csv')) + list(HEADLINES_DIR.glob('*.jsonl'))
print(f'\nHeadline files in {HEADLINES_DIR}: {len(headline_files)}')
for f in headline_files[:5]:
    print(f'  {f.name}')
if not headline_files:
    print('  (none found — see note below about corpus options)')


In [ ]:
# ── B2: Load or build headline corpus ─────────────────────────────────────────
# The notebook accepts headlines in two formats:
#   CSV : columns [date, headline]   (date = YYYY-MM-DD or YYYY-MM)
#   JSONL: one JSON per line with keys 'date' and 'text'
#
# GDELT GKG filter for US-China themes (alternative to manual download):
#   gdelt.gdeltproject.org/data/gdeltv2/masterfilelist.txt
#   Theme filter: RELATIONS_US_CHINA, ECON_TRADE_DISPUTE, MILITARY_POSTURE

HEADLINE_CACHE = PROC_NLP / 'headlines_loaded.parquet'

def load_headline_corpus(directory: Path) -> pd.DataFrame:
    """
    Load all headline CSV/JSONL files from directory.
    Returns DataFrame with columns: date (Timestamp), text (str).
    """
    frames = []
    for f in directory.glob('*.csv'):
        try:
            df = pd.read_csv(f, usecols=lambda c: c.lower() in ('date','headline','text','title'))
            df.columns = [c.lower() for c in df.columns]
            if 'headline' in df.columns:
                df = df.rename(columns={'headline': 'text'})
            if 'title' in df.columns:
                df = df.rename(columns={'title': 'text'})
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            frames.append(df[['date','text']].dropna())
        except Exception as e:
            print(f'  Could not load {f.name}: {e}')

    for f in directory.glob('*.jsonl'):
        try:
            import jsonlines
            with jsonlines.open(f) as reader:
                rows = [{'date': pd.Timestamp(r['date']), 'text': r['text']} for r in reader]
            frames.append(pd.DataFrame(rows))
        except Exception as e:
            print(f'  Could not load {f.name}: {e}')

    if not frames:
        return pd.DataFrame(columns=['date','text'])
    return pd.concat(frames, ignore_index=True).dropna(subset=['text'])


if HEADLINE_CACHE.exists():
    df_headlines = pd.read_parquet(HEADLINE_CACHE)
    print(f'Loaded headline cache: {len(df_headlines):,} rows')
else:
    df_headlines = load_headline_corpus(HEADLINES_DIR)
    if not df_headlines.empty:
        df_headlines.to_parquet(HEADLINE_CACHE)
    print(f'Loaded {len(df_headlines):,} headlines from {len(headline_files)} files')

if df_headlines.empty:
    print()
    print('No headline corpus found. Two options:')
    print('  1. Place CSV/JSONL files in:', HEADLINES_DIR)
    print('  2. Run Section B3-GDELT to use GDELT GKG tone as the sentiment proxy.')
    print()
    print('Section B will produce NaN columns that are excluded from controls automatically.')


In [ ]:
# ── B3: FinBERT sentiment pipeline ────────────────────────────────────────────
# If transformers is available and we have a headline corpus, run FinBERT.
# If not, fall back to GDELT AvgTone as the sentiment signal.

SENTIMENT_OUT = PROC_NLP / 'sentiment_monthly.csv'

def run_finbert(texts: list, batch_size: int = 32) -> pd.DataFrame:
    """
    Run ProsusAI/finbert on a list of texts.
    Returns DataFrame with columns: pos, neg, neu (probabilities).
    """
    from transformers import pipeline
    device = 0 if HAS_GPU else -1  # 0 = first GPU, -1 = CPU
    model_name = 'ProsusAI/finbert'
    print(f'Loading {model_name} on {"GPU" if HAS_GPU else "CPU"}...')
    nlp_pipe = pipeline('text-classification', model=model_name,
                        top_k=None, device=device, truncation=True, max_length=512)

    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        out   = nlp_pipe(batch)
        for item in out:
            scores = {d['label'].lower(): d['score'] for d in item}
            results.append({
                'pos': scores.get('positive', np.nan),
                'neg': scores.get('negative', np.nan),
                'neu': scores.get('neutral',  np.nan),
            })
        if i % 500 == 0:
            print(f'  Processed {i}/{len(texts)} headlines', end='\r')

    return pd.DataFrame(results)


if SENTIMENT_OUT.exists():
    df_sent = pd.read_csv(SENTIMENT_OUT, index_col=0, parse_dates=True)
    df_sent.index = pd.to_datetime(df_sent.index).to_period('M').to_timestamp()
    print(f'Loaded cached sentiment: {df_sent.shape}')

elif not df_headlines.empty and HAS_TRANSFORMERS:
    print('Running FinBERT on headline corpus...')
    scores = run_finbert(df_headlines['text'].tolist())
    df_headlines_scored = df_headlines.copy().reset_index(drop=True)
    df_headlines_scored[['finbert_pos','finbert_neg','finbert_neu']] = scores.values
    df_headlines_scored['finbert_net'] = (
        df_headlines_scored['finbert_pos'] - df_headlines_scored['finbert_neg']
    )
    df_headlines_scored['month'] = df_headlines_scored['date'].dt.to_period('M').dt.to_timestamp()

    df_sent = df_headlines_scored.groupby('month').agg(
        finbert_pos = ('finbert_pos', 'mean'),
        finbert_neg = ('finbert_neg', 'mean'),
        finbert_net = ('finbert_net', 'mean'),
        finbert_vol = ('finbert_net', 'std'),   # intra-month dispersion (Müller 2022)
    )
    df_sent.index.name = None
    df_sent.to_csv(SENTIMENT_OUT)
    print(f'FinBERT sentiment saved: {SENTIMENT_OUT}')

else:
    # ── Fallback: use GDELT AvgTone as sentiment proxy ──────────────────────
    # GDELT AvgTone is computed from article text using a dictionary method.
    # It is a coarser measure than FinBERT but has full 1990-2022 coverage.
    # Document in thesis: "Sentiment is proxied by GDELT AvgTone; FinBERT
    # scores were not available for the full sample period."
    print('FALLBACK: No headline corpus or transformers — using GDELT tone as sentiment proxy.')
    if 'gdelt_tone_mean' in df_gdelt.columns:
        df_sent = df_gdelt[['gdelt_tone_mean']].rename(
            columns={'gdelt_tone_mean': 'finbert_net'}
        )
        df_sent['finbert_pos'] = np.nan   # not available in fallback
        df_sent['finbert_neg'] = np.nan
        df_sent['finbert_vol'] = np.nan
        df_sent.to_csv(SENTIMENT_OUT)
        print('Saved GDELT-based sentiment proxy to:', SENTIMENT_OUT)
        print('NOTE: finbert_pos, finbert_neg, finbert_vol are NaN in fallback mode.')
        print('      Only finbert_net (= GDELT AvgTone) has coverage.')
    else:
        df_sent = pd.DataFrame(index=date_range,
                               columns=['finbert_pos','finbert_neg','finbert_net','finbert_vol'])
        print('No sentiment data available. All sentiment columns will be NaN.')

print()
if not df_sent.empty:
    print(df_sent.describe().round(4).to_string())


---
## Section C — Topic Model Proportions (LDA / BERTopic)

**Why topic models:** Sentiment scores collapse text into a scalar. Topic models
reveal *what* is being discussed — whether the dominant theme is trade, military,
diplomacy, or economics. Monthly topic proportions add a qualitative dimension that
complements the PRI index.

**The professor's plan specifies:** LDA or BERTopic on news headlines.

**Implementation strategy:**
- If `headline corpus` is available → train LDA (sklearn, CPU-safe) or BERTopic (GPU preferred)
- If no corpus → use GDELT CAMEO codes as a topic proxy:
  - CAMEO 3–8 (cooperation) → `topic_coop`
  - CAMEO 18–20 (assault/hostility) → `topic_conflict`
  - CAMEO 9–14 (diplomatic) → `topic_diplomatic`
  - This is a rule-based topic proxy with full 1990-2022 coverage

**Number of topics:** 5 topics (conflict, trade/economy, military, diplomacy, other).
This is consistent with Saadaoui's narrative about geopolitical turning points
involving distinct domains.


In [ ]:
# ── C1: Detect BERTopic ────────────────────────────────────────────────────────
try:
    import bertopic
    HAS_BERTOPIC = True
    print(f'BERTopic available: {bertopic.__version__}')
except ImportError:
    HAS_BERTOPIC = False
    print('BERTopic NOT available.')
    print('Install with: pip install bertopic')
    print('Falling back to LDA (sklearn) or GDELT CAMEO codes.')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


In [ ]:
# ── C2: Train topic model ──────────────────────────────────────────────────────
TOPICS_OUT = PROC_NLP / 'topics_monthly.csv'

# Topic labels: we pre-assign semantic labels by inspecting top words.
# The actual assignment is done after training by examining top terms per topic.
# Adjust TOPIC_LABELS after inspecting the model output.
N_TOPICS = 5
TOPIC_LABELS = {
    0: 'topic_conflict',
    1: 'topic_trade',
    2: 'topic_military',
    3: 'topic_diplomatic',
    4: 'topic_other',
}

def train_lda(df_hl: pd.DataFrame, n_topics: int = 5) -> tuple:
    """
    Train LDA on headline texts. Returns (model, vectorizer, doc-topic matrix).
    """
    vectorizer = CountVectorizer(
        max_df=0.95, min_df=5,
        stop_words='english',
        max_features=5000,
        ngram_range=(1, 2),
    )
    dtm   = vectorizer.fit_transform(df_hl['text'].fillna(''))
    model = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method='online',
        max_iter=20,
        n_jobs=-1,
    )
    doc_topics = model.fit_transform(dtm)
    return model, vectorizer, doc_topics


def print_top_words(model, vectorizer, n_top: int = 10):
    words = vectorizer.get_feature_names_out()
    print('Top words per LDA topic (assign TOPIC_LABELS manually after inspection):')
    for i, comp in enumerate(model.components_):
        top = comp.argsort()[:-n_top-1:-1]
        print(f'  Topic {i}: {" | ".join(words[top])}')


if TOPICS_OUT.exists():
    df_topics = pd.read_csv(TOPICS_OUT, index_col=0, parse_dates=True)
    df_topics.index = pd.to_datetime(df_topics.index).to_period('M').to_timestamp()
    print(f'Loaded cached topics: {df_topics.shape}')

elif not df_headlines.empty:
    if HAS_BERTOPIC:
        # ── BERTopic (preferred) ───────────────────────────────────────────────
        from bertopic import BERTopic
        print('Training BERTopic...')
        topic_model = BERTopic(
            nr_topics=N_TOPICS,
            calculate_probabilities=True,
            verbose=False,
        )
        topics_raw, probs = topic_model.fit_transform(df_headlines['text'].tolist())
        print('BERTopic top words:')
        print(topic_model.get_topic_info().head(10).to_string())

        # Build monthly proportions from per-document probabilities
        df_hl_t = df_headlines.copy().reset_index(drop=True)
        df_hl_t['month'] = df_hl_t['date'].dt.to_period('M').dt.to_timestamp()
        prob_df = pd.DataFrame(probs, columns=[f't{i}' for i in range(probs.shape[1])])
        df_hl_t = pd.concat([df_hl_t, prob_df], axis=1)
        grp_cols = [f't{i}' for i in range(probs.shape[1])]
        df_topics = df_hl_t.groupby('month')[grp_cols].mean()
        df_topics.columns = [TOPIC_LABELS.get(i, f'topic_{i}') for i in range(len(grp_cols))]
    else:
        # ── LDA fallback (sklearn, CPU) ────────────────────────────────────────
        print('Training LDA (sklearn)...')
        lda_model, vec, doc_topics = train_lda(df_headlines, n_topics=N_TOPICS)
        print_top_words(lda_model, vec)
        print()
        print('>>> IMPORTANT: Review top words above and update TOPIC_LABELS dict <<<')
        print('    The labels (conflict/trade/military/diplomatic/other) are placeholders.')
        print('    Reassign after visual inspection of top words.')

        df_hl_t = df_headlines.copy().reset_index(drop=True)
        df_hl_t['month'] = df_hl_t['date'].dt.to_period('M').dt.to_timestamp()
        topic_df_raw = pd.DataFrame(
            doc_topics,
            columns=[TOPIC_LABELS.get(i, f'topic_{i}') for i in range(N_TOPICS)]
        )
        df_hl_t = pd.concat([df_hl_t, topic_df_raw], axis=1)
        df_topics = df_hl_t.groupby('month')[list(TOPIC_LABELS.values())].mean()

    df_topics.to_csv(TOPICS_OUT)
    print(f'Topics saved: {TOPICS_OUT}')

else:
    # ── GDELT CAMEO rule-based topic proxy ─────────────────────────────────────
    # Without a headline corpus, GDELT CAMEO codes provide rule-based topic proxies.
    # CAMEO root codes: 3-8 = coop, 9-14 = diplomatic, 18-20 = conflict
    # This has full 1990-2022 coverage and is methodologically defensible:
    # CAMEO codes are expert-coded event classifications, not NLP model outputs.
    # Document in thesis as "rule-based topic proxy from GDELT CAMEO classification."
    print('FALLBACK: No headline corpus — using GDELT CAMEO codes as topic proxy.')
    if not df_gdelt.empty and 'gdelt_conflict_share' in df_gdelt.columns:
        df_topics = df_gdelt[['gdelt_conflict_share','gdelt_coop_share']].copy()
        df_topics = df_topics.rename(columns={
            'gdelt_conflict_share': 'topic_conflict',
            'gdelt_coop_share'    : 'topic_coop',
        })
        df_topics['topic_trade']      = np.nan  # not separable from CAMEO alone
        df_topics['topic_military']   = np.nan
        df_topics['topic_diplomatic'] = np.nan
        df_topics.to_csv(TOPICS_OUT)
        print('CAMEO-based topic proxy saved. NLP-derived topics (trade, military,')
        print('diplomatic) are NaN — requires headline corpus for those.')
    else:
        df_topics = pd.DataFrame(index=date_range,
                                 columns=list(TOPIC_LABELS.values()))
        print('No topic data available. All topic columns will be NaN.')

print()
if not df_topics.empty:
    print(df_topics.describe().round(4).to_string())


---
## Section D — Merge NLP Features with Feature Matrix

All NLP features are merged onto the existing macro feature matrix from Step 1.2a.

**Lag convention:** All NLP features are lagged 1 month before merging.
This matches the convention in `02_feature_matrix_v3.ipynb` and prevents look-ahead bias:
at estimation time *t*, the model sees sentiment / topic proportions from month *t−1*.

**`var_roles_nlp.json`** extends `var_roles.json` with new role categories:
- `controls_sentiment`: FinBERT scores (or GDELT tone fallback)
- `controls_topics`: topic proportions
- `controls_gdelt`: raw GDELT event-level features
- `controls_all_nlp_dense`: sentiment + GDELT (full coverage)
- `controls_all_nlp`: all NLP features (may have sparse coverage)


In [ ]:
# # ── D1: Prepare all NLP series ─────────────────────────────────────────────────
# def prep_series(df: pd.DataFrame, lag: int = 1) -> pd.DataFrame:
#     """Align index to month-start, lag by `lag` months, restrict to sample window."""
#     df = df.copy()
#     df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
#     df = df.reindex(date_range)
#     df = df.shift(lag)
#     return df.loc[START:END]

# nlp_frames = {}

# # GDELT
# if not df_gdelt.empty:
#     nlp_frames['gdelt'] = prep_series(df_gdelt)
#     print(f'GDELT    : {nlp_frames["gdelt"].notna().sum().sum()} total non-NaN values')

# # Sentiment
# if not df_sent.empty:
#     nlp_frames['sentiment'] = prep_series(df_sent)
#     print(f'Sentiment: {nlp_frames["sentiment"].notna().sum().sum()} total non-NaN values')

# # Topics
# if not df_topics.empty:
#     nlp_frames['topics'] = prep_series(df_topics)
#     print(f'Topics   : {nlp_frames["topics"].notna().sum().sum()} total non-NaN values')


In [ ]:
# # ── D2: Merge onto base feature matrix ────────────────────────────────────────
# df_nlp = df_base.copy()
# df_nlp.index = pd.to_datetime(df_nlp.index).to_period('M').to_timestamp()

# for name, frame in nlp_frames.items():
#     for col in frame.columns:
#         df_nlp[col] = frame[col]

# print(f'NLP-enriched matrix: {df_nlp.shape[0]} obs × {df_nlp.shape[1]} columns')
# print(f'New NLP columns: {sum(1 for k,v in nlp_frames.items() for _ in v.columns)}')


In [ ]:
# # ── D3: Coverage audit ─────────────────────────────────────────────────────────
# nlp_cols = [c for frame in nlp_frames.values() for c in frame.columns]
# nlp_cols = [c for c in nlp_cols if c in df_nlp.columns]

# audit_nlp = pd.DataFrame({
#     'n_obs'     : df_nlp[nlp_cols].notna().sum(),
#     'pct_filled': (df_nlp[nlp_cols].notna().mean() * 100).round(1),
#     'first_obs' : df_nlp[nlp_cols].apply(lambda s: s.first_valid_index()),
#     'last_obs'  : df_nlp[nlp_cols].apply(lambda s: s.last_valid_index()),
# }).sort_values('pct_filled', ascending=False)

# print(audit_nlp.to_string())

# zero_nlp   = audit_nlp[audit_nlp['pct_filled'] == 0].index.tolist()
# sparse_nlp = audit_nlp[(audit_nlp['pct_filled'] > 0) & (audit_nlp['pct_filled'] < 80)].index.tolist()
# dense_nlp  = audit_nlp[audit_nlp['pct_filled'] >= 80].index.tolist()

# print(f'\nDense NLP (≥80%): {dense_nlp}')
# print(f'Sparse NLP (<80%): {sparse_nlp}')
# print(f'Zero coverage   : {zero_nlp}')


In [ ]:
# # ── D4: Variable roles ────────────────────────────────────────────────────────
# # Load the macro var_roles from Step 1.2a and extend with NLP roles.
# macro_roles_path = PROC / 'var_roles.json'
# if macro_roles_path.exists():
#     with open(macro_roles_path) as f:
#         VAR_ROLES = json.load(f)
#     print(f'Loaded macro var_roles: {list(VAR_ROLES.keys())}')
# else:
#     VAR_ROLES = {}
#     print('WARNING: var_roles.json not found — rebuild from scratch')

# # NLP role additions
# nlp_gdelt_cols = [c for c in nlp_frames.get('gdelt', pd.DataFrame()).columns
#                   if c in dense_nlp + sparse_nlp]
# nlp_sent_cols  = [c for c in nlp_frames.get('sentiment', pd.DataFrame()).columns
#                   if c in dense_nlp + sparse_nlp]
# nlp_topic_cols = [c for c in nlp_frames.get('topics', pd.DataFrame()).columns
#                   if c in dense_nlp + sparse_nlp]

# VAR_ROLES['controls_gdelt']     = [c for c in nlp_gdelt_cols if c not in zero_nlp]
# VAR_ROLES['controls_sentiment'] = [c for c in nlp_sent_cols  if c not in zero_nlp]
# VAR_ROLES['controls_topics']    = [c for c in nlp_topic_cols if c not in zero_nlp]

# # Dense NLP spec: ≥80% coverage only
# VAR_ROLES['controls_all_nlp_dense'] = (
#     VAR_ROLES.get('controls_all_ml_dense', [])
#     + [c for c in nlp_gdelt_cols + nlp_sent_cols + nlp_topic_cols if c in dense_nlp]
# )

# # Full NLP spec: all non-zero coverage
# VAR_ROLES['controls_all_nlp'] = (
#     VAR_ROLES.get('controls_all_ml', [])
#     + [c for c in nlp_gdelt_cols + nlp_sent_cols + nlp_topic_cols if c not in zero_nlp]
# )

# VAR_ROLES['EXCLUDED_nlp_zero_coverage'] = zero_nlp

# print('\nNLP variable roles:')
# for k in ['controls_gdelt','controls_sentiment','controls_topics',
#           'controls_all_nlp_dense','controls_all_nlp']:
#     print(f'  {k:30s} ({len(VAR_ROLES[k]):2d}): {VAR_ROLES[k]}')


In [ ]:
# # ── D5: Safety assertions and export ──────────────────────────────────────────
# # 1. All controls_all_nlp_dense columns must exist
# missing_dense = [c for c in VAR_ROLES['controls_all_nlp_dense']
#                  if c not in df_nlp.columns]
# assert not missing_dense, f'controls_all_nlp_dense references missing cols: {missing_dense}'
# print('✓ All controls_all_nlp_dense columns present')

# # 2. All controls_all_nlp columns must exist
# missing_all = [c for c in VAR_ROLES['controls_all_nlp']
#                if c not in df_nlp.columns]
# assert not missing_all, f'controls_all_nlp references missing cols: {missing_all}'
# print('✓ All controls_all_nlp columns present')

# # 3. No forbidden raw columns
# for must_be_gone in ['lbrent','cny_usd','em_fx_idx','reer','indpro','epu','bdi']:
#     assert must_be_gone not in df_nlp.columns, \
#         f'{must_be_gone} leaked into NLP feature matrix'
# print('✓ No forbidden raw columns')

# # Export
# df_nlp.to_csv(OUT_CSV)
# with open(OUT_ROLES, 'w') as fh:
#     json.dump({k: list(v) for k, v in VAR_ROLES.items()}, fh, indent=2)

# print(f'\nSaved: {OUT_CSV}  ({df_nlp.shape})')
# print(f'Saved: {OUT_ROLES}')

# # Summary
# print(f'\nFinal feature matrix summary:')
# print(f'  Total columns          : {df_nlp.shape[1]}')
# print(f'  Macro controls (dense) : {len(VAR_ROLES.get("controls_all_ml_dense",[]))}')
# print(f'  NLP controls (dense)   : {len([c for c in VAR_ROLES["controls_all_nlp_dense"] if c not in VAR_ROLES.get("controls_all_ml_dense",[])])}')
# print(f'  Total dense controls   : {len(VAR_ROLES["controls_all_nlp_dense"])}')
# print(f'\nUse var_roles_nlp.json → controls_all_nlp_dense for the primary DoubleML spec.')
# print(f'Use var_roles_nlp.json → controls_all_nlp for robustness checks.')


---
## Thesis Documentation Notes

These are the methodological points to address in the thesis write-up for Step 1.2b.

### On the sentiment measure
- **Primary:** FinBERT (Huang et al. 2023) outputs three probabilities per sentence (positive,
  negative, neutral). Monthly aggregation: mean of daily net sentiment (`pos − neg`) and its
  standard deviation (`finbert_vol`, capturing intra-month disagreement, Müller 2022).
- **Fallback:** GDELT AvgTone (dictionary-based) when a headline corpus is unavailable.
  Acknowledge the methodological difference: FinBERT is a transformer fine-tuned on
  financial text; GDELT tone uses the General Inquirer dictionary.
- **`finbert_vol`** is the standard deviation of daily net sentiment within each month.
  Müller (2022, *Journal of Finance*) shows that sentiment dispersion predicts asset price
  volatility beyond mean sentiment levels — this is the "event novelty" proxy in the plan.

### On topic models
- **LDA hyperparameters:** 5 topics, `max_iter=20`, `online` learning. The number of topics
  was chosen to match the conceptual categories in Saadaoui's narrative (conflict,
  trade, military, diplomatic, other). Validate with perplexity and coherence scores.
- **BERTopic** (if used): uses HDBSCAN clustering over sentence-transformer embeddings.
  Cite Grootendorst (2022). GPU strongly recommended for corpora > 10K documents.
- **GDELT CAMEO proxy:** If using rule-based topic proxies, cite Schrodt (2012) for the
  CAMEO coding scheme. Note that CAMEO codes reflect event type, not text semantics.

### On GDELT event counts
- GDELT 1.0 logs events from English-language print media. Coverage is uneven before 1995
  due to smaller media footprint in the data. Consider restricting GDELT-based inference
  to post-1995 in robustness checks.
- The Goldstein scale (Goldstein 1992) maps CAMEO codes to a cooperation–conflict continuum
  from −10 to +10. It is an expert-coded measure, not a statistical estimate.

### Stationarity
- `gdelt_tone_mean` and `gdelt_goldstein_mean`: persistent but bounded; treat as stationary.
- `finbert_net`: stationary by construction (bounded probability difference).
- `gdelt_n_events`: may be trending upward (GDELT coverage grew over time). Take log or
  first-difference if ADF test rejects stationarity.
- Topic proportions: bounded [0,1] and sum to 1; treat as stationary.

### Cite
- Huang, A. H., Wang, H., & Yang, Y. (2023). FinBERT: A Large Language Model for Extracting
  Information from Financial Text. *Contemporary Accounting Research*.
- Müller, M. (2022). Textual sentiment, option characteristics, and stock return predictability.
  *Journal of Finance*.
- Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure.
  *arXiv:2203.05794*.
- Schrodt, P. A. (2012). CAMEO: Conflict and Mediation Event Observations Event and Actor
  Codebook. *Pennsylvania State University*.
